# Ascend 算子优化 Agent

先在含 `torch_npu` 和 `triton-ascend` 的 NPU 镜像中，将本项目目录上传并解压。支持 vector add / RMSNorm / GDN verify / GDN commit 的固定模板配置调优；所有候选必须经过实卡验证。

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / "ascend_optimizer").is_dir(), "请从项目目录打开此 Notebook"
subprocess.run([sys.executable, "-m", "ascend_optimizer", "doctor"], check=True)


## 先运行无需模型密钥的调优

每次新实验使用新的工作区。已有工作区直接继续 search，不重复 init。

In [ ]:
workspace = ROOT / "runs" / "notebook-rms-search"
if not (workspace / "task.json").exists():
    subprocess.run([sys.executable, "-m", "ascend_optimizer", "init", "--task", "rms_norm", "--workspace", str(workspace)], check=True)
subprocess.run([sys.executable, "-m", "ascend_optimizer", "search", "--workspace", str(workspace), "--max-candidates", "4", "--profile"], check=True)


In [ ]:
from IPython.display import Markdown, display
display(Markdown((workspace / "REPORT.md").read_text()))


## 接入你自己的 OpenAI 兼容模型

API key 通过隐藏输入读取，不保存到 Notebook 源码或输出。地址、模型名按你实际开通的服务填写，模型需支持工具调用。

In [ ]:
import getpass
os.environ["ASCEND_OPT_BASE_URL"] = input("API base URL（服务商指定根路径）: ").strip()
os.environ["ASCEND_OPT_MODEL"] = input("模型名称: ").strip()
os.environ["ASCEND_OPT_API_MODE"] = input("协议 chat / responses（默认 chat）: ").strip() or "chat"
os.environ["OPENAI_API_KEY"] = getpass.getpass("API key（隐藏输入）: ")


In [ ]:
llm_workspace = ROOT / "runs" / "notebook-rms-llm"
if not (llm_workspace / "task.json").exists():
    subprocess.run([sys.executable, "-m", "ascend_optimizer", "init", "--task", "rms_norm", "--workspace", str(llm_workspace)], check=True)
subprocess.run([sys.executable, "-m", "ascend_optimizer", "agent", "--workspace", str(llm_workspace), "--max-steps", "24", "--max-tools", "64", "--profile"], check=True)
display(Markdown((llm_workspace / "REPORT.md").read_text()))


实验结束后下载 `runs` 目录或打包取回本机。Notebook 会话有时间限制，不能依赖临时磁盘长期保留源码和结果。不要同时运行两项 NPU 性能实验。

## 投机解码项目 GDN

在新工作区调优冻结项目的 GDN verify / commit；密钥通过上面的隐藏输入配置。真实模型运行至少完成三项候选裁决。输出保留原始配对时间、逐位验证和模型 usage，不代表整模型吞吐。

In [ ]:
gdn_workspace = ROOT / "runs" / "notebook-gdn-verify"
if not (gdn_workspace / "task.json").exists():
    subprocess.run([sys.executable, "-m", "ascend_optimizer", "init", "--task", "gdn_verify", "--workspace", str(gdn_workspace)], check=True)
subprocess.run([sys.executable, "-m", "ascend_optimizer", "agent", "--workspace", str(gdn_workspace), "--max-steps", "40", "--max-tools", "96", "--timeout", "900", "--profile"], check=True)
display(Markdown((gdn_workspace / "REPORT.md").read_text()))
